# G3 Parity — Vector Field on Three-Sample Plane

Visualizes score `s(x,σ) = (D-x)/σ²` and denoiser `D(x,σ)` on a 2D plane spanned by:
- **x_a** (cyan ★) — a training sample (memorized attractor)
- **x_b** (green ★) — a valid non-training sample: Hamming=2 from x_a (2 bits flipped within group-0, parity preserved)
- **x_c** (red ★) — an invalid sample: Hamming=1 from **both** x_a and x_b (single bit flip → group-0 parity broken)

Plane axes (true L2 distance, equal scale on both axes):
- α-axis (L2): direction x_a → x_b;  α=0 → x_a,  α=‖x_b-x_a‖ → x_b
- β-axis (L2): component of x_c − x_a perpendicular to α;  β=‖ac_perp‖ → x_c

## Checkpoints (G3 rep2)
| Checkpoint | mem ratio | mid-σ train/test gap | Interpretation |
|---|---|---|---|
| ep 58780  | ~0%  | ~0      | post rule-learning, pre-memorization |
| ep 242446 | ~1%  | 0.013   | train/test gap just starting |
| ep 345511 | ~1%  | 0.067   | gap open, pre-mem onset |
| ep 492388 | ~8%  | 0.134   | just after memorization onset |
| ep 701704 | ~21% | 0.159   | well into memorization |


In [ ]:
import sys
sys.path.insert(0, '/n/home12/binxuwang/Github/DiffusionAttnConsistency')

import numpy as np
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['axes.spines.top']   = False
matplotlib.rcParams['axes.spines.right'] = False
import matplotlib.pyplot as plt

from core.vector_field_lib import (
    load_model, load_training_data,
    eval_field_on_grid, make_plane_hash,
    project_to_basis,
    plot_vector_field_2d,
    plot_denoiser_target_2d,
)

SAVEROOT = '/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning'
FIGDIR   = '/n/home12/binxuwang/Github/DiffusionAttnConsistency/figures/vector_field'
import os; os.makedirs(FIGDIR, exist_ok=True)

EXP        = 'DiT_mini_parity_N4096_D36_G3_even_rep2'
GROUP_SIZE = 3
DEVICE     = 'cpu'   # change to 'cuda' if available

## 1. Build the three anchor samples and 2D plane

In [ ]:
x_train = load_training_data(EXP, saveroot=SAVEROOT)   # (4096, 36)

# x_a: training sample (memorized attractor)
x_a = x_train[0].numpy().copy()

# x_b: valid non-training — flip 2 bits in group 0 ((-1)²=+1 → parity preserved), Hamming=2 from x_a
x_b = x_a.copy(); x_b[0] *= -1; x_b[1] *= -1
assert np.prod(x_b[:3]) == 1.0,  "group-0 parity violated"
assert (x_a != x_b).sum() == 2,  "Hamming(x_a, x_b) should be 2"

# x_c: invalid — flip 1 bit from x_a (breaks group-0 parity), Hamming=1 from BOTH x_a and x_b
# Flipping bit-0 from x_a: x_c agrees with x_b on bit-0, disagrees on bit-1 → Hamming(x_c,x_b)=1 ✓
x_c = x_a.copy(); x_c[0] *= -1
assert np.prod(x_c[:3]) == -1.0, "group-0 should be invalid"
assert (x_a != x_c).sum() == 1,  "Hamming(x_a, x_c) should be 1"
assert (x_b != x_c).sum() == 1,  "Hamming(x_b, x_c) should be 1"

print(f"x_a: {x_a[:6]}")
print(f"x_b: {x_b[:6]}  Hamming(a,b)={(x_a!=x_b).sum()}  valid")
print(f"x_c: {x_c[:6]}  Hamming(a,c)={(x_a!=x_c).sum()}  Hamming(b,c)={(x_b!=x_c).sum()}  invalid (group-0 parity={np.prod(x_c[:3]):.0f})")


In [ ]:
# ── Plane basis (unit vectors → L2-scaled axes) ───────────────────────────────
ab = (x_b - x_a).astype(np.float32)
L_ab = float(np.linalg.norm(ab))
v_ab = ab / L_ab                                        # unit α-direction

ac      = (x_c - x_a).astype(np.float32)
ac_perp = ac - ac.dot(v_ab) * v_ab                      # perpendicular component
L_perp  = float(np.linalg.norm(ac_perp))
v_ac    = ac_perp / L_perp                              # unit β-direction

# Coordinates of the three anchors in (α_L2, β_L2) — these ARE the plot coordinates now
xa_coord = (0.0, 0.0)
xb_coord = (L_ab, 0.0)
xc_alpha = float(ac.dot(v_ab))                          # L2 projection onto α
xc_beta  = L_perp                                       # L2 distance in β direction
xc_coord = (xc_alpha, xc_beta)

print(f"L_ab={L_ab:.3f}  L_perp={L_perp:.3f}")
print(f"Plane coords (L2):  x_a=(0,0)  x_b=({L_ab:.2f},0)  x_c=({xc_alpha:.2f},{xc_beta:.2f})")
print(f"Hamming: a-b={(x_a!=x_b).sum()}  a-c={(x_a!=x_c).sum()}  b-c={(x_b!=x_c).sum()}")

# ── Grid — equal L2 scale on both axes ────────────────────────────────────────
NGRID   = 45
margin  = 0.4 * L_ab
alpha_ax = np.linspace(-margin, L_ab + margin, NGRID, dtype=np.float32)
beta_ax  = np.linspace(-margin, L_ab + margin, NGRID, dtype=np.float32)
A, B    = np.meshgrid(alpha_ax, beta_ax, indexing='ij')   # (NGRID, NGRID) L2 coords

grid_x = (x_a[None,None,:]
           + A[:,:,None] * v_ab[None,None,:]
           + B[:,:,None] * v_ac[None,None,:]).astype(np.float32)
print(f"Grid: {grid_x.shape}  α=[{alpha_ax[0]:.2f},{alpha_ax[-1]:.2f}]  β=[{beta_ax[0]:.2f},{beta_ax[-1]:.2f}]")

PLANE_HASH = make_plane_hash(x_a, x_b, x_c)
RANGE_TAG  = f"a{alpha_ax[0]:.2f}_{alpha_ax[-1]:.2f}"
CACHE_DIR  = os.path.join(SAVEROOT, EXP, 'vector_field_cache')


## 2. Helper: plot one checkpoint × all σ values

In [ ]:
def plot_ckpt(model, ckpt_label, sigmas=(0.2, 0.5, 1.0, 2.0),
              save_tag=None, quiver_scale=None):
    """Two-row figure: score magnitude (top) + denoiser D·v_ab (bottom) for each σ."""
    s = 4  # quiver stride
    grid_sp = (alpha_ax[-1] - alpha_ax[0]) / len(alpha_ax)
    fig, axes = plt.subplots(2, len(sigmas), figsize=(4.5*len(sigmas), 9.5))
    fig.suptitle(f'G3 rep2  {ckpt_label}\n'
                 f'Plane: x_a(train) – x_b(valid novel) – x_c(invalid)  [L2 scale]',
                 fontsize=10, fontweight='bold')

    for col, sigma in enumerate(sigmas):
        res  = eval_field_on_grid(model, grid_x, sigma, device=DEVICE,
                                   cache_dir=CACHE_DIR, cache_key=f'ep{ckpt_label}_{PLANE_HASH}_{RANGE_TAG}')
        u_s, v_s = project_to_basis(res['score'],  v_ab, v_ac)
        Du,  _   = project_to_basis(res['D'],      v_ab, v_ac)
        # D_pull = D(x,σ) - x = σ²·score; use directly (avoids x·v_ab coordinate confusion)
        disp_u, disp_v = project_to_basis(res['D_pull'], v_ab, v_ac)

        # ── top: score magnitude + arrows ────────────────────────────────────
        ax  = axes[0][col]
        mag = res['mag_score']
        vmax = np.percentile(mag, 95)
        im  = ax.pcolormesh(alpha_ax, beta_ax, mag.T, cmap='magma',
                            vmin=0, vmax=vmax, shading='auto', rasterized=True)
        plt.colorbar(im, ax=ax, shrink=0.75, label='‖score‖')
        ax.quiver(A[::s,::s], B[::s,::s], u_s[::s,::s], v_s[::s,::s],
                  color='white', alpha=0.85,
                  scale=vmax/(1.5*grid_sp), scale_units='xy', angles='xy', width=0.004)
        ax.plot(0,        0,        'c*', ms=13, label='x_a (train)',       zorder=10, mec='k', mew=0.5)
        ax.plot(L_ab,     0,        'g*', ms=13, label='x_b (valid novel)', zorder=10, mec='k', mew=0.5)
        ax.plot(xc_alpha, xc_beta,  'r*', ms=13, label='x_c (invalid)',     zorder=10, mec='k', mew=0.5)
        ax.plot(xc_alpha, -xc_beta, 'r^', ms=10, label='x_d (mirror)',      zorder=10, mec='k', mew=0.5, alpha=0.6)
        ax.set_xlim(alpha_ax[0], alpha_ax[-1]); ax.set_ylim(beta_ax[0], beta_ax[-1])
        ax.set_aspect('equal')
        ax.set_xlabel('α (L2)', fontsize=9); ax.set_ylabel('β (L2)', fontsize=9)
        ax.set_title(f'σ={sigma:.2f}  ‖score‖', fontsize=9)
        ax.axvline(0, color='gray', lw=0.4, ls='--', alpha=0.4)
        ax.axhline(0, color='gray', lw=0.4, ls='--', alpha=0.4)
        if col == 0: ax.legend(fontsize=7, loc='upper left', framealpha=0.7)

        # ── bottom: denoiser D·v_ab + (D−x) arrows ───────────────────────────
        # D−x = σ²·score → arrows parallel to top, scaled by σ²
        ax = axes[1][col]
        clim = np.percentile(np.abs(Du), 97)
        im2 = ax.pcolormesh(alpha_ax, beta_ax, Du.T, cmap='RdBu_r',
                            vmin=-clim, vmax=clim, shading='auto', rasterized=True)
        plt.colorbar(im2, ax=ax, shrink=0.75, label='D·v_ab')
        dscale = np.percentile(np.sqrt(disp_u**2+disp_v**2), 90) / (1.5*grid_sp)
        ax.quiver(A[::s,::s], B[::s,::s], disp_u[::s,::s], disp_v[::s,::s],
                  color='k', alpha=0.75, scale=dscale, scale_units='xy', angles='xy', width=0.004)
        ax.plot(0,        0,        'c*', ms=13, zorder=10, mec='k', mew=0.5)
        ax.plot(L_ab,     0,        'g*', ms=13, zorder=10, mec='k', mew=0.5)
        ax.plot(xc_alpha, xc_beta,  'r*', ms=13, zorder=10, mec='k', mew=0.5)
        ax.plot(xc_alpha, -xc_beta, 'r^', ms=10, zorder=10, mec='k', mew=0.5, alpha=0.6)
        ax.set_xlim(alpha_ax[0], alpha_ax[-1]); ax.set_ylim(beta_ax[0], beta_ax[-1])
        ax.set_aspect('equal')
        ax.set_xlabel('α (L2)', fontsize=9)
        ax.set_ylabel('β (L2)', fontsize=9)
        ax.set_title(f'σ={sigma:.2f}  D·v_ab  (arrows: D−x=σ²·score)', fontsize=9)
        ax.axvline(0, color='gray', lw=0.4, ls='--', alpha=0.4)
        ax.axhline(0, color='gray', lw=0.4, ls='--', alpha=0.4)

    plt.tight_layout()
    if save_tag:
        tag = EXP.replace('DiT_mini_', '')
        for ext in ['png', 'pdf']:
            fig.savefig(f'{FIGDIR}/vf_v3_{save_tag}.{ext}', dpi=150, bbox_inches='tight')
    return fig


## 3. ep 58780 — post rule-learning, pre-memorization

In [ ]:
m58k, _, _ = load_model(EXP, 58780, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m58k, 'ep 58780  (post rule-learning, pre-mem  |  mem~0%, gap~0)',
                save_tag='ep58780')
plt.show()

## 4. ep 242446 — train/test gap just starting

In [ ]:
m242k, _, _ = load_model(EXP, 242446, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m242k, 'ep 242446  (gap just starting  |  mem~1%, gap=0.013)',
                save_tag='ep242446')
plt.show()

## 5. ep 345511 — gap open, pre-memorization onset

In [ ]:
m345k, _, _ = load_model(EXP, 345511, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m345k, 'ep 345511  (gap open, pre-mem  |  mem~1%, gap=0.067)',
                save_tag='ep345511')
plt.show()

## 6. ep 492388 — just after memorization onset

In [ ]:
m492k, _, _ = load_model(EXP, 492388, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m492k, 'ep 492388  (just after mem onset  |  mem=8%, gap=0.134)',
                save_tag='ep492388')
plt.show()

## 7. ep 701704 — well into memorization

In [ ]:
m701k, _, _ = load_model(EXP, 701704, device=DEVICE, saveroot=SAVEROOT)
fig = plot_ckpt(m701k, 'ep 701704  (well into memorization  |  mem=21%, gap=0.159)',
                save_tag='ep701704')
plt.show()

## 8. Single-σ comparison across all checkpoints

In [ ]:
# Side-by-side score magnitude at σ=0.5 for all 5 checkpoints
ckpt_models = [
    ('ep 58k\npre-mem',    m58k),
    ('ep 242k\ngap start', m242k),
    ('ep 345k\ngap open',  m345k),
    ('ep 492k\nmem onset', m492k),
    ('ep 701k\nmem 21%',   m701k),
]
sigma_sel = 0.5
s = 5

fig, axes = plt.subplots(2, len(ckpt_models), figsize=(4*len(ckpt_models), 8))
fig.suptitle(f'G3 rep2  σ={sigma_sel}  — score & denoiser across checkpoints', fontsize=11, fontweight='bold')

for col, (lbl, model) in enumerate(ckpt_models):
    res = eval_field_on_grid(model, grid_x, sigma_sel, device=DEVICE)
    u_s, v_s = project_to_basis(res['score'], v_ab, v_ac)
    Du, _    = project_to_basis(res['D'],     v_ab, v_ac)
    disp_u, disp_v = project_to_basis(res['D_pull'], v_ab, v_ac)  # D-x=σ²·score

    ax = axes[0][col]
    mag = res['mag_score']
    im = ax.pcolormesh(alpha_ax, beta_ax, mag.T, cmap='hot',
                       vmin=0, vmax=np.percentile(mag, 97), shading='auto', rasterized=True)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.quiver(A[::s,::s], B[::s,::s], u_s[::s,::s], v_s[::s,::s],
              color='white', alpha=0.7, scale=30)
    ax.plot(0, 0, 'c*', ms=12, zorder=10)
    ax.plot(L_ab, 0, 'g*', ms=12, zorder=10)
    ax.plot(xc_alpha, xc_beta, 'r*', ms=12, zorder=10)
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel('α', fontsize=8); ax.set_ylabel('β', fontsize=8)
    if col == 0: ax.set_ylabel('Score mag\nβ', fontsize=8)

    ax = axes[1][col]
    clim = max(abs(Du.min()), abs(Du.max()))
    im2 = ax.pcolormesh(alpha_ax, beta_ax, Du.T, cmap='RdBu_r',
                        vmin=-clim, vmax=clim, shading='auto', rasterized=True)
    plt.colorbar(im2, ax=ax, shrink=0.8)
    ax.quiver(A[::s,::s], B[::s,::s], (Du-A)[::s,::s], (Dv-B)[::s,::s],
              color='k', alpha=0.5, scale=20)
    ax.plot(0, 0, 'c*', ms=12, zorder=10)
    ax.plot(L_ab, 0, 'g*', ms=12, zorder=10)
    ax.plot(xc_alpha, xc_beta, 'r*', ms=12, zorder=10)
    ax.set_xlabel('α', fontsize=8)
    if col == 0: ax.set_ylabel('Denoiser D·v_ab\nβ', fontsize=8)

plt.tight_layout()
for ext in ['pdf', 'png']:
    fig.savefig(f'{FIGDIR}/G3_checkpoint_comparison_sigma{sigma_sel}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()